In [1]:
from platform import python_version
print(python_version())

3.11.14


### Cluster with Tahoe or sc-GTP

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/single_cell'), True)

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/single_cell/deconv.h5ad (6.8 MB)
1604


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [12]:
res.cell_type_expression("Ductal cell type 1").shape

(1604, 153)

In [13]:
res.cell_type_expression("Ductal cell type 2").shape

(1604, 153)

### 2. theta is now fixed -> expand Z to every gene

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        force=force, verbose=verbose)

force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_singc / fname_ad
compression = "gzip"
# adata.write_h5ad(filename_ad, compression=compression)
print(f"AData saved as {filename_ad},  ({filename_ad.stat().st_size/1e6:.0f} MB), compressed with {compression}")


verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file
Table opened ((27169, 153)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_matrix.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_metadata.tsv'
57,530 cells x 24,005 genes | obs: []
AData saved as /home/flavio/uv/perturb_agent/data/single_cell/count-matrix.h5ad,  (338 MB), compressed with gzip
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 24

In [15]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)

In [16]:
dic = {}

for cell_state in res.states:
    print(cell_state)
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z

Fibroblast cell
Stellate cell
Macrophage cell
Endothelial cell
T cell
B cell
Ductal cell type 2
Endocrine cell
Ductal cell type 1
Acinar cell


In [17]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

Fibroblast cell


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
A1BG,6.551e-01,4.418e-01,6.680e-01,2.693e-01,4.914e-01,4.378e-01,1.043e+00,8.103e-01,9.495e-01,7.379e-01,...,NaN,0.772,0.328,NaN,NaN,NaN,1.515e-01,NaN,2.648e-01,NaN
A1BG-AS1,3.150e+00,1.251e+00,1.249e+00,2.249e+00,1.408e+00,1.848e+00,3.725e+00,1.988e+00,2.017e+00,2.263e+00,...,NaN,6.944,9.022,NaN,NaN,NaN,7.544e+00,NaN,1.126e+00,NaN
A1CF,3.463e+00,1.384e+00,2.188e+00,1.346e-01,1.305e+00,8.334e-02,1.176e+00,2.169e+00,1.218e+00,3.142e+00,...,NaN,1.033,3.148,NaN,NaN,NaN,1.032e+01,NaN,1.701e+00,NaN
A2M,1.344e+03,1.258e+03,1.165e+03,8.277e+02,1.477e+03,8.272e+02,1.130e+03,1.373e+03,2.035e+03,2.286e+03,...,NaN,1083.582,1060.576,NaN,NaN,NaN,2.068e+03,NaN,2.003e+03,NaN
A2M-AS1,6.254e+00,7.056e+00,2.835e+00,2.690e+00,5.443e+00,7.466e+00,6.568e+00,7.522e+00,6.423e+00,1.268e+01,...,NaN,2.626,7.625,NaN,NaN,NaN,1.319e+01,NaN,2.975e+00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,8.548e-09,2.341e-08,6.967e-08,7.866e-09,2.045e-08,2.942e-08,6.440e-08,2.001e-08,1.809e-07,1.226e-08,...,NaN,0.021,0.000,NaN,NaN,NaN,7.822e-08,NaN,6.635e-09,NaN
ZYG11B,1.060e+02,1.070e+02,1.151e+02,8.354e+01,1.278e+02,9.075e+01,1.159e+02,1.173e+02,9.685e+01,1.466e+02,...,NaN,53.810,83.431,NaN,NaN,NaN,1.003e+02,NaN,5.187e+01,NaN
ZYX,7.451e+01,5.875e+01,7.764e+01,1.232e+02,4.028e+01,7.744e+01,5.810e+01,6.771e+01,7.150e+01,5.684e+01,...,NaN,87.890,61.471,NaN,NaN,NaN,3.394e+01,NaN,3.417e+02,NaN
ZZEF1,1.460e+02,1.129e+02,9.541e+01,1.231e+02,1.044e+02,1.199e+02,1.055e+02,1.418e+02,1.251e+02,1.251e+02,...,NaN,129.316,99.746,NaN,NaN,NaN,5.434e+02,NaN,8.607e+01,NaN


### Ductal 2 - malignant

In [18]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")

In [19]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

FAM83A-AS1 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXA10-AS {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
HOXB-AS3 {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}
MIR7-3HG {'Fibroblast cell': nan, 'Stellate cell': nan, 'Macrophage cell': nan}


In [20]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [21]:
[g for g in prog1 if g in df_bulk.index]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'HOXB-AS4', 'MIR7-3HG']

### present in the scRNA reference?

In [22]:
  
[g for g in prog1 if g in ref.columns]

['FAM83A-AS1', 'HOXA10-AS', 'HOXB-AS3', 'MIR7-3HG']

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [23]:
set(ref.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [24]:
set(s2t.index.to_list())

{'Acinar cell',
 'B cell',
 'Ductal cell type 1',
 'Ductal cell type 2',
 'Endocrine cell',
 'Endothelial cell',
 'Fibroblast cell',
 'Macrophage cell',
 'Stellate cell',
 'T cell'}

In [25]:
adata.obs

,cluster,cell_type,cell_state
cell,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell
...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1


In [26]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))

tissue
normal    11
tumor     24
Name: sample, dtype: int64
tissue              normal  tumor
cell_state                       
Acinar cell           1423    512
B cell                  31   2416
Ductal cell type 1    7671   2646
Ductal cell type 2       0  11315
Endocrine cell         270    459
Endothelial cell      3983   5134
Fibroblast cell        940   5802
Macrophage cell        559   4802
Stellate cell          623   5284
T cell                  44   3616


### Count Malignant Cells - accordingo to transcriptomics

In [27]:
d2 = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

57530


cell
T1_AAACCTGAGATGTCGG    False
T1_AAACGGGGTCATGCAT    False
T1_AAAGATGCATGTTGAC    False
T1_AAAGATGGTCGAGTTT    False
T1_AAAGATGGTCTCTCTG    False
Name: cell_state, dtype: bool

In [28]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

41986


In [29]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

,cluster,cell_type,cell_state,sample,tissue
cell,,,,,
T1_AAACCTGAGATGTCGG,Fibroblast cell,Fibroblast cell,Fibroblast cell,T1,tumor
T1_AAACGGGGTCATGCAT,Stellate cell,Stellate cell,Stellate cell,T1,tumor
T1_AAAGATGCATGTTGAC,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCGAGTTT,Macrophage cell,Macrophage cell,Macrophage cell,T1,tumor
T1_AAAGATGGTCTCTCTG,Endothelial cell,Endothelial cell,Endothelial cell,T1,tumor
...,...,...,...,...,...
N11_TTTGCGCGTGCGCTTG,Endothelial cell,Endothelial cell,Endothelial cell,N11,normal
N11_TTTGGTTCATTGAGCT,Acinar cell,Acinar cell,Acinar cell,N11,normal
N11_TTTGGTTGTCCGACGT,Ductal cell type 1,Ductal cell type 1,Ductal cell type 1,N11,normal


In [30]:
from collections import Counter

Counter(adata.obs["cell_state"] )

Counter({'Malignant ductal': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [31]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

Counter({'malignant': 11315,
         'Ductal cell type 1': 10317,
         'Endothelial cell': 9117,
         'Fibroblast cell': 6742,
         'Stellate cell': 5907,
         'Macrophage cell': 5361,
         'T cell': 3660,
         'B cell': 2447,
         'Acinar cell': 1935,
         'Endocrine cell': 729})

In [32]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

{'Fibroblast cell': 'Fibroblast cell',
 'Stellate cell': 'Stellate cell',
 'Macrophage cell': 'Macrophage cell',
 'Endothelial cell': 'Endothelial cell',
 'T cell': 'T cell',
 'B cell': 'B cell',
 'Malignant ductal': 'malignant',
 'Endocrine cell': 'Endocrine cell',
 'Ductal cell type 1': 'Ductal cell type 1',
 'Acinar cell': 'Acinar cell'}

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [33]:
res.__dict__.keys()

dict_keys(['theta', 'theta_stage1', 'theta_type', 'tumor_purity', 'genes', 'Z', 'states'])

In [34]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

### Prism programs

In [35]:
Z_full, genes_full = prism.full_Z(res, df_bulk, ref)

### MalignantCluster

In [36]:
# del(MalignantCluster)

In [37]:
from libs.prism_malig_lib import MalignantCluster

In [38]:
type(res)

libs.prism_lib.DeconvResult

In [39]:
cbio.root_mprog_disease

PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD')

In [40]:
root_mprog_cluster = create_dir(cbio.root_mprog_disease, 'cluster')

cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True
'''
mc = MalignantCluster(prism=prism, res=res, df_bulk=df_bulk, ref=ref, 
                      root_mprog_cluster=root_mprog_cluster, 
                      organ="Pancreas", cell_name=cell_name, cell_types=None)
'''

import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

0.23.1


In [41]:
mc = pml.MalignantCluster(prism, res, df_bulk, ref, root_mprog_cluster, organ="Pancreas")

X, diag = mc.prepare_malignant_matrix(decouple_purity=False, keep_genes=mc.program1_panel, drop_pattern=r"^N-")
print(X.shape)
X.head(3)

excluded 22/153 samples by keep_samples/drop_pattern
(117, 2000)


,A1CF,AACS,AADAC,AATK,ABAT,ABCA12,ABCA7,ABCB9,ABCC3,ABCC6,...,ZNF774,ZNF787,ZNF792,ZNF816,ZNF888,ZNRF1,ZNRF2,ZSCAN29,ZSWIM5,ZWINT
T-C3L-02890,5.982,6.258,5.175,5.251,5.758,4.525,6.916,5.286,8.488,3.813,...,3.765,5.004,5.415,5.472,6.948,4.979,5.810,6.161,3.973,5.391
T-C3L-03635,4.757,5.724,1.766,4.362,5.829,6.391,5.053,3.572,9.914,2.841,...,4.566,4.348,5.478,6.968,7.315,4.711,6.219,6.158,3.293,5.177
T-C3L-02701,5.355,6.172,2.614,5.341,5.771,4.948,6.295,3.261,9.049,3.479,...,4.707,4.696,4.896,5.616,6.857,4.948,5.456,6.081,4.002,5.462


In [42]:
lista = [x for x in X.index if x.startswith('T-')]
X.shape[0], len(lista) == X.shape[0]

(117, True)

In [43]:
diag.keys()

dict_keys(['samples_excluded_by_filter', 'samples_dropped', 'n_genes_expressed', 'n_genes_share_not_computable', 'n_genes_share_ok', 'forced_genes_status', 'n_genes_kept', 'n_hvg', 'pc_theta_pearson_raw', 'pc_theta_pearson', 'decouple_purity', 'pc_theta_note', 'sample_mean_expr', 'sample_total_Z', 'theta_mal', 'n_samples_used', 'theta_excluded', 'theta_kept'])

In [44]:
diag["samples_excluded_by_filter"]

['N-C3L-04072',
 'N-C3L-00589',
 'N-C3L-03123',
 'N-C3L-04080',
 'N-C3L-00640',
 'N-C3N-01719',
 'N-C3L-07033',
 'N-C3L-00819',
 'N-C3L-07032',
 'N-C3L-01689',
 'N-C3N-01899',
 'N-C3N-00517',
 'N-C3N-03069',
 'N-C3N-02765',
 'N-C3L-07037',
 'N-C3N-02589',
 'N-C3N-02996',
 'N-C3L-02606',
 'N-C3N-03173',
 'N-C3N-02696',
 'N-TCGA-H6-8124',
 'N-TCGA-H6-A45N']

In [45]:
diag["pc_theta_pearson"] 

[-0.5957714445145382,
 -0.2052535675488754,
 0.22715677713490706,
 -0.3591881167228313,
 -0.1580137862705503]

In [46]:
diag["pc_theta_pearson_raw"]   # PC-vs-theta on logx (pre-decoupling)

[-0.5958663540532598,
 -0.2048870348093191,
 0.2275776260666874,
 -0.3592621819717635,
 -0.15951815005740053]

In [47]:
diag["pc_theta_note"]          # warns the decoupled version is ~0 by construction

'decouple_purity=False, so pc_theta_pearson and pc_theta_pearson_raw are the same matrix and both are informative: a large |r| on an early PC means the clustering is tracking tumour purity.'

In [48]:
diag["sample_mean_expr"]       # Xc.mean(axis=1) per sample

T-C3L-02890       6.188
T-C3L-03635       6.036
T-C3L-02701       6.146
T-C3L-04072       5.692
T-C3L-00589       6.043
                  ...  
T-TCGA-2L-AAQM    4.473
T-TCGA-3A-A9IR    4.186
T-TCGA-3A-A9IV    4.605
T-TCGA-2J-AABT    6.059
T-TCGA-H6-A45N    6.240
Length: 117, dtype: float32

In [49]:
diag["sample_total_Z"]         # ms.Z.sum(axis=1) per sample

T-C3L-02890       1.000e+06
T-C3L-03635       1.000e+06
T-C3L-02701       1.000e+06
T-C3L-04072       1.000e+06
T-C3L-00589       1.000e+06
                    ...    
T-TCGA-2L-AAQM    1.000e+06
T-TCGA-3A-A9IR    1.000e+06
T-TCGA-3A-A9IV    1.000e+06
T-TCGA-2J-AABT    1.000e+06
T-TCGA-H6-A45N    1.000e+06
Length: 117, dtype: float32

In [50]:
diag["theta_excluded"]

count    15.000
mean      0.291
std       0.418
min       0.000
25%       0.002
50%       0.060
75%       0.562
max       0.985
Name: Ductal cell type 2, dtype: float64

In [51]:
diag["theta_kept"]

count    130.000
mean       0.347
std        0.254
min        0.000
25%        0.145
50%        0.312
75%        0.493
max        1.000
Name: Ductal cell type 2, dtype: float64

### Inspecting vars

In [52]:
import inspect
print(pml.__version__)
print("drop_pattern" in inspect.signature(mc.prepare_malignant_matrix).parameters)

0.23.1
True


In [53]:
info = mc.inspect_de_schema(genes=X.columns)
print(info.keys())
info["columns"]

dict_keys(['shard_file', 'file_mb', 'total_rows_in_shard', 'columns', 'dtypes', 'head', 'distinct_gene_name', 'n_distinct_gene_name', 'distinct_baseMean', 'n_distinct_baseMean', 'distinct_log2FoldChange', 'n_distinct_log2FoldChange', 'distinct_lfcSE', 'n_distinct_lfcSE', 'distinct_stat', 'n_distinct_stat', 'distinct_pvalue', 'n_distinct_pvalue', 'distinct_padj', 'n_distinct_padj', 'distinct_plate', 'n_distinct_plate', 'distinct_n_cells_trt', 'n_distinct_n_cells_trt', 'distinct_n_cells_ctrl', 'n_distinct_n_cells_ctrl', 'distinct_Cell_ID_Cellosaur', 'n_distinct_Cell_ID_Cellosaur', 'distinct_Cell_ID_DepMap', 'n_distinct_Cell_ID_DepMap', 'distinct_drug', 'n_distinct_drug', 'distinct_concentration', 'n_distinct_concentration', 'distinct_concentration_unit', 'n_distinct_concentration_unit', 'distinct_Cell_Name_Vevo', 'n_distinct_Cell_Name_Vevo', 'cell_line_metadata_columns', 'MATCH cell_line_metadata.Cell_ID_Cellosaur -> DE.Cell_ID_Cellosaur', 'MATCH cell_line_metadata.cell_name -> DE.Cell_N

['gene_name',
 'baseMean',
 'log2FoldChange',
 'lfcSE',
 'stat',
 'pvalue',
 'padj',
 'plate',
 'n_cells_trt',
 'n_cells_ctrl',
 'Cell_ID_Cellosaur',
 'Cell_ID_DepMap',
 'drug',
 'concentration',
 'concentration_unit',
 'Cell_Name_Vevo']

In [54]:
info["matches"]

['MATCH cell_line_metadata.Cell_ID_Cellosaur -> DE.Cell_ID_Cellosaur',
 'MATCH cell_line_metadata.cell_name -> DE.Cell_Name_Vevo',
 'MATCH query genes -> DE.gene_name']

In [55]:
info["dtypes"]

{'gene_name': 'object',
 'baseMean': 'float32',
 'log2FoldChange': 'float32',
 'lfcSE': 'float32',
 'stat': 'float32',
 'pvalue': 'float32',
 'padj': 'float32',
 'plate': 'object',
 'n_cells_trt': 'int64',
 'n_cells_ctrl': 'int64',
 'Cell_ID_Cellosaur': 'object',
 'Cell_ID_DepMap': 'object',
 'drug': 'object',
 'concentration': 'float32',
 'concentration_unit': 'object',
 'Cell_Name_Vevo': 'object'}

In [56]:
info["resolved_columns"]

{'gene': 'gene_name',
 'stat': 'stat',
 'cell_line': 'Cell_ID_Cellosaur',
 'drug': 'drug'}

In [57]:
info["numeric_profile"]      # min / max / mean / frac_negative / n_unique


,min,max,mean,frac_negative,n_unique
baseMean,0.000,95136.398,36.460,0.000,69136
log2FoldChange,-4.858,7.180,0.081,0.232,68458
lfcSE,0.007,4.425,1.248,0.000,68449
stat,-46.548,73.326,0.019,0.232,68488
pvalue,0.000,1.000,0.470,0.000,68433
padj,0.000,1.000,0.586,0.000,23253
n_cells_trt,1378.000,2165.000,1745.045,0.000,4
n_cells_ctrl,4862.000,4862.000,4862.000,0.000,1
concentration,0.050,0.050,0.050,0.000,1


In [58]:
info["signed_candidates"]

['log2FoldChange', 'stat']

In [59]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]

(50, 102)

In [60]:
cov["block_probe_counts"]        # min probes per block

count    50.0
mean      3.4
std       0.5
min       3.0
25%       3.0
50%       3.0
75%       4.0
max       4.0
Name: count, dtype: float64

In [61]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

(52, ['CVCL_0025', 'CVCL_0031', 'CVCL_0039', 'CVCL_0060', 'CVCL_0078'])

In [62]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

(50, 102)

### Shards - A database shard, or simply a shard, is a horizontal partition of data within a database or search engine.

In [63]:
cvcl = mc.organ_cell_lines() 
mc.shard_index(stride=6)          # ~120 new footers, ~9 min (not 172)
shards = mc.find_shards_for(cvcl) # expect: located 11, missing 0

shard index: 172 cached, 0 to read (stride=6, 1026 shards total)
targets: 11 | located: 7 | missing: 4
shards to scan: 237/1026 (23%)  ~17 min at 4.3 s/shard
  NOT located: ['CVCL_0186', 'CVCL_1634', 'CVCL_1638', 'CVCL_1639']
  index resolves 50 distinct Cell_ID_Cellosaur values from 172 probes (gap~6).
  -> run index_coverage(): if these are absent from the DE table, no stride will find them and you should proceed without them.


In [64]:
len(cvcl), cvcl[:3]

(11, ['CVCL_0152', 'CVCL_0186', 'CVCL_0334'])

In [65]:
dropped = ('CVCL_0186','CVCL_1634','CVCL_1638','CVCL_1639')
cvcl7  = [c for c in cvcl if c not in dropped]
shards = mc.find_shards_for(cvcl7)          # expect missing: 0

targets: 7 | located: 7 | missing: 0
shards to scan: 237/1026 (23%)  ~17 min at 4.3 s/shard


In [66]:
mc.shard_sizes(shards)

,shard,bytes
0,metadata/pseudobulk_differential_expression/train-00096-of-01026.parquet,92513713
1,metadata/pseudobulk_differential_expression/train-00097-of-01026.parquet,91964715
2,metadata/pseudobulk_differential_expression/train-00098-of-01026.parquet,92490710
3,metadata/pseudobulk_differential_expression/train-00099-of-01026.parquet,92183301
4,metadata/pseudobulk_differential_expression/train-00100-of-01026.parquet,88160979
...,...,...
232,metadata/pseudobulk_differential_expression/train-00920-of-01026.parquet,83238171
233,metadata/pseudobulk_differential_expression/train-00921-of-01026.parquet,80567798
234,metadata/pseudobulk_differential_expression/train-00922-of-01026.parquet,85303011
235,metadata/pseudobulk_differential_expression/train-00923-of-01026.parquet,89114559


In [67]:
mc.shard_sizes(shards)["bytes"].sum() / 1e9      # GB for the 237 shards

20.990016063

In [68]:
# to much: lets paralelize
# mc.download_shards(shards, dry_run=True)
mc.download_shards(shards, max_workers=8)

237 shards, 20.99 GB (median 90 MB/shard)


Fetching 237 files: 100%|██████████| 237/237 [00:00<00:00, 423.87it/s]


In [69]:
"""
df["condition"] = df["cell_line_id"].astype(str) + "|" + df["drug"].astype(str)
df_pivot = (df.pivot(index="gene", columns="condition", values="stat")
        .astype(dtype))
"""

df_pivot, cond = mc.load_tahoe_de(genes=X.columns.to_list(), organs=("Pancreas",),
                                  mode="download", _shard_subset=shards, force=True)

Fetching 4 files: 100%|██████████| 4/4 [00:02<00:00,  1.76it/s]


In [70]:
print(df_pivot.shape)                                  # genes x conditions
df_pivot.head(8)

(1986, 2653)


condition,CVCL_0152|(R)-Verapamil (hydrochloride),CVCL_0152|(S)-Crizotinib,CVCL_0152|18β-Glycyrrhetinic acid,CVCL_0152|4EGI-1,CVCL_0152|5-Azacytidine,CVCL_0152|5-Fluorouracil,CVCL_0152|8-Hydroxyquinoline,CVCL_0152|9-ING-41,CVCL_0152|APTO-253,CVCL_0152|AT7519,...,CVCL_C466|Vortioxetine,CVCL_C466|XRK3F2,CVCL_C466|Zileuton,CVCL_C466|c-Kit-IN-1,CVCL_C466|crizotinib,CVCL_C466|olaparib,CVCL_C466|palbociclib,CVCL_C466|venetoclax,CVCL_C466|vincristine,CVCL_C466|γ-Oryzanol
gene,,,,,,,,,,,,,,,,,,,,,
A1CF,-0.466,0.383,-1.177,-1.062,-2.607,-1.135,0.060,-0.030,-0.668,0.689,...,0.149,-0.350,0.464,-0.076,-0.403,0.655,-2.550e-04,-0.048,0.723,1.146
AACS,1.064,-0.747,-0.181,0.564,0.458,0.089,0.664,-0.240,-0.316,0.969,...,-0.025,0.513,0.087,1.004,-0.070,-0.717,1.189e+00,0.121,0.867,-0.323
AADAC,-1.989,-0.931,-2.890,0.720,0.705,-0.181,-2.254,-3.302,0.457,-2.357,...,-0.268,-0.414,-0.749,0.410,-0.035,0.312,-8.589e-01,-0.785,0.170,0.220
AATK,-1.528,-0.021,0.608,-1.022,-0.063,1.126,-0.244,0.228,-0.989,0.116,...,-0.235,0.956,0.324,1.715,0.234,0.951,1.560e-01,0.641,-0.297,-0.003
ABAT,1.206,-0.828,-0.354,1.710,-0.696,-0.142,0.776,0.256,1.692,-0.156,...,0.591,1.612,-0.891,0.215,-1.004,0.633,4.112e-01,-0.837,-0.672,0.369
ABCA12,0.653,0.446,-0.005,1.170,0.720,0.768,-1.095,-0.373,0.827,0.706,...,0.199,-0.110,0.323,0.261,0.539,1.025,4.309e-01,0.839,1.100,0.599
ABCA7,0.078,-1.072,0.061,-1.091,-2.009,-0.626,-1.363,-1.342,-0.200,1.450,...,-0.727,-1.334,-0.821,0.770,0.311,0.344,-6.434e-01,0.086,0.027,0.087
ABCB9,1.334,-0.058,0.240,0.332,-0.341,-0.332,0.751,0.229,1.756,1.265,...,1.336,-0.934,-1.418,0.215,0.230,0.487,-8.274e-01,-0.293,-1.111,-0.331


In [71]:
cond.head(3)

,cell_line_id,drug,Cell_ID_Cellosaur,cell_name,Organ,moa-fine,moa-broad,targets,human-approved
condition,,,,,,,,,
CVCL_0152|(R)-Verapamil (hydrochloride),CVCL_0152,(R)-Verapamil (hydrochloride),CVCL_0152,AsPC-1,Pancreas,unclear,inhibitor/antagonist,ABCB1,no
CVCL_0152|(R)-Verapamil (hydrochloride),CVCL_0152,(R)-Verapamil (hydrochloride),CVCL_0152,AsPC-1,Pancreas,unclear,inhibitor/antagonist,ABCB1,no
CVCL_0152|(R)-Verapamil (hydrochloride),CVCL_0152,(R)-Verapamil (hydrochloride),CVCL_0152,AsPC-1,Pancreas,unclear,inhibitor/antagonist,ABCB1,no


In [72]:
cond["cell_line_id"].nunique(), cond["drug"].nunique()

(7, 379)

In [73]:
cond["cell_line_id"].value_counts()     # expect 7 lines

cell_line_id
CVCL_0152    2274
CVCL_0428    2274
CVCL_1635    2274
CVCL_0480    1895
CVCL_0334    1516
CVCL_1119    1137
CVCL_C466     379
Name: count, dtype: int64

In [74]:
cond["drug"].value_counts()


drug
(R)-Verapamil (hydrochloride)    31
Orlistat                         31
Palmatine (chloride)             31
Paclitaxel                       31
PH-797804                        31
                                 ..
Docetaxel (Trihydrate)           31
Docetaxel                        31
Diphenhydramine                  31
Dinaciclib                       31
γ-Oryzanol                       31
Name: count, Length: 379, dtype: int64

In [75]:
print("\n".join(np.unique(cond["drug"])))

(R)-Verapamil (hydrochloride)
(S)-Crizotinib
18β-Glycyrrhetinic acid
4EGI-1
5-Azacytidine
5-Fluorouracil
8-Hydroxyquinoline
9-ING-41
APTO-253
AT7519
AZD-7648
AZD-8055
AZD1390
AZD2858
Abemaciclib
Abiraterone acetate
Acetazolamide
Acetohexamide
Adagrasib
Adenine
Adenosine
Afatinib
Aliskiren
Allantoin
Allopurinol
Almonertinib (hydrochloride)
Almonertinib (mesylate)
Alpelisib
Altretamine
Amsacrine
Anastrozole
Anethole trithione
Apalutamide
Aprepitant
Arbutin
Artemether
Artesunate
Asciminib
Aspirin
Ataluren
Atazanavir (sulfate)
Auranofin
Azithromycin (hydrate)
BAY1125976
BI-3406
BI-78D3
Baicalin
Balsalazide (sodium hydrate)
Belinostat
Belumosudil
Belumosudil (mesylate)
Belzutifan
Bendamustine
Benproperine (phosphate)
Bentamapimod
Benztropine (mesylate)
Berbamine
Berbamine (dihydrochloride)
Berberine (chloride hydrate)
Bergenin
Bestatin
Bestatin (hydrochloride)
Betamethasone dipropionate
Bexarotene
Bicalutamide
Bimatoprost
Bimiralisib
Binimetinib
Bisoprolol (hemifumarate)
Bortezomib
Bosentan

In [76]:
## 2D-values, stacked distribution
df_pivot.stack().describe()

count    5.232e+06
mean    -4.649e-02
std      1.698e+00
min     -5.551e+01
25%     -6.898e-01
50%     -5.990e-03
75%      6.468e-01
max      1.451e+02
dtype: float64

In [77]:
np.sum(np.sum(df_pivot<=1))

4382435

In [78]:
np.sum(np.sum(df_pivot<=-1))

928141

In [79]:
float((df_pivot <= -1).mean().mean())

0.17615600951857122

In [80]:
(df_pivot >= 1).mean(axis=0)

condition
CVCL_0152|(R)-Verapamil (hydrochloride)    0.149
CVCL_0152|(S)-Crizotinib                   0.128
CVCL_0152|18β-Glycyrrhetinic acid          0.123
CVCL_0152|4EGI-1                           0.152
CVCL_0152|5-Azacytidine                    0.140
                                           ...  
CVCL_C466|olaparib                         0.245
CVCL_C466|palbociclib                      0.127
CVCL_C466|venetoclax                       0.163
CVCL_C466|vincristine                      0.174
CVCL_C466|γ-Oryzanol                       0.123
Length: 2653, dtype: float64

In [81]:
(df_pivot >= 1).mean(axis=1)

gene
A1CF       0.080
AACS       0.231
AADAC      0.079
AATK       0.140
ABAT       0.199
           ...  
ZNRF1      0.435
ZNRF2      0.230
ZSCAN29    0.435
ZSWIM5     0.163
ZWINT      0.454
Length: 1986, dtype: float64

### Cluster

#### All three checks pass

- 0.4989 negative confirms stat is a genuinely signed statistic 
- the WTCS sign convention is sound. 
- 1986 of 2000 HVGs found in Tahoe is 99.3% coverage, better than I expected for a Parse 3' assay.

#### Score it:

In [82]:
cc = mc.consensus_cluster(X)
cc

{2: {'labels': T-C3L-02890       2
  T-C3L-03635       2
  T-C3L-02701       2
  T-C3L-04072       2
  T-C3L-00589       2
                   ..
  T-TCGA-2L-AAQM    1
  T-TCGA-3A-A9IR    1
  T-TCGA-3A-A9IV    1
  T-TCGA-2J-AABT    2
  T-TCGA-H6-A45N    2
  Name: k2, Length: 117, dtype: int32,
  'consensus':                 T-C3L-02890  T-C3L-03635  T-C3L-02701  T-C3L-04072  T-C3L-00589  T-C3L-03123  \
  T-C3L-02890             1.0          1.0          1.0          1.0          1.0          1.0   
  T-C3L-03635             1.0          1.0          1.0          1.0          1.0          1.0   
  T-C3L-02701             1.0          1.0          1.0          1.0          1.0          1.0   
  T-C3L-04072             1.0          1.0          1.0          1.0          1.0          1.0   
  T-C3L-00589             1.0          1.0          1.0          1.0          1.0          1.0   
  ...                     ...          ...          ...          ...          ...          ...   
  T-TCG

### PAC

Proportion of Ambiguous Clustering — a measure of how decisively the consensus clustering assigns samples, from Șenbabaoğlu et al. (2014). 

It's how choose_k picks k.

The consensus matrix C[i,j] is the fraction of resamples in which samples i and j landed in the same cluster, given both were drawn. 

Perfect structure gives values of 0 or 1 — pairs always together or always apart. Unstable structure gives values scattered in between.

PAC is just the fraction of pairs sitting in that ambiguous middle:



In [83]:
consensus = cc[2]['consensus']
consensus.iloc[:5, :10]


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439
T-C3L-02890,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
T-C3L-03635,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
T-C3L-02701,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
T-C3L-04072,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
T-C3L-00589,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [84]:
def _pac(consensus, lo=0.1, hi=0.9):
    v = consensus[np.triu_indices_from(consensus, k=1)]
    return float(((v > lo) & (v < hi)).mean())

_pac(consensus.values, lo=0.1, hi=0.9)

0.0

Low PAC = crisp, reproducible partition. choose_k takes the smallest k within pac_tol of the minimum, preferring parsimony when several k are comparably stable.

Why it misled you here. PAC rewards reproducibility, not biological meaning, and those come apart badly for unbalanced splits. Peeling six outliers off 119 samples is maximally reproducible — every resample isolates them identically — so PAC approaches zero. The metric is behaving exactly as designed while pointing at nothing interesting.

It's also mechanically biased toward small k, since fewer clusters means fewer boundaries to disagree about. That's why choose_k at k=2 deserves scepticism rather than confidence on its own.

So read the diagnostics together:

In [85]:
dfa = pd.DataFrame({k: {"pac": v["pac"], "coph": v["cophenetic"],
                    "sil": v["silhouette"],
                    "sizes": v["labels"].value_counts().tolist()}
                    for k, v in cc.items()}).T

dfa

,pac,coph,sil,sizes
2,0.0,1.0,1.0,"[111, 6]"
3,0.093,0.999,0.98,"[109, 6, 2]"
4,0.016,1.0,0.966,"[106, 6, 3, 2]"
5,0.009,1.0,0.988,"[62, 44, 6, 3, 2]"
6,0.058,1.0,0.959,"[62, 44, 5, 3, 2, 1]"
7,0.122,0.998,0.874,"[61, 44, 5, 3, 2, 1, 1]"
8,0.136,0.994,0.887,"[62, 38, 5, 5, 3, 2, 1, 1]"


The sizes column is the one that would have caught this. 

A k with low PAC and balanced clusters is trustworthy; 
low PAC with a 6/119 split is an outlier detector. 

Cophenetic correlation and silhouette are worth glancing at too, though both share the same blind spot — none of them knows the difference between a real subtype and six weird samples.

In [86]:
k = mc.choose_k(cc)
k

2

### k=2 is the expected answer for PDAC 

- Moffitt's classical vs basal-like is a two-group axis. 
- The earlier problem wasn't k=2, it was the 6-vs-119 split. 
- So the question now is what the two groups are.

Three checks, in order of how much they'd change your interpretation:

In [87]:
labels = cc[2]["labels"]
print('n counts', labels.value_counts().to_dict())          # balanced now?
print("")
print(diag["pc_theta_pearson"])                 # PC1 vs theta_mal
print("")
print(pd.crosstab(labels, pd.Series(
    ['TCGA' if 'TCGA' in s else 'CPTAC' for s in X.index], index=X.index)))

n counts {2: 111, 1: 6}

[-0.5957714445145382, -0.2052535675488754, 0.22715677713490706, -0.3591881167228313, -0.1580137862705503]

col_0  CPTAC  TCGA
k2                
1          0     6
2         44    67


A near-even split with |r| below ~0.3 on PC1 is what you want.

- If the crosstab shows the split tracking TCGA vs CPTAC, it's a batch axis — plausible given your strandedness history, 
- and it would mean the unstranded harmonisation didn't fully remove the cohort effect.

Then the test that actually names the clusters:

In [88]:
basal = ["KRT81","KRT5","KRT6A","KRT17","S100A2","SPRR3","TP63","DHRS9","VGLL1"]
clas  = ["GATA6","TFF1","TFF2","TFF3","LGALS4","CLDN18","CEACAM6","AGR2", "ANXA10","REG4","CTSE","MUC13"]
Z = (X - X.mean()) / X.std()
sc = pd.DataFrame({
    "basal":     Z[[g for g in basal if g in X.columns]].mean(axis=1),
    "classical": Z[[g for g in clas  if g in X.columns]].mean(axis=1)})
print(sc.groupby(labels).mean().round(2))

    basal  classical
k2                  
1   -1.86      -2.02
2    0.10       0.11


If **one cluster is basal-high/classical-low** and **the other the reverse**, you've recovered Moffitt in the deconvolved malignant compartment

- which is a genuinely stronger result than the bulk clustering you started with, 
- because it's not confounded by stromal content. That was the whole point of the deconvolution detour.

If instead both clusters co-elevate the two programs, 
- you're seeing the same cellularity axis as before, and the purity decoupling didn't clear it.

Note how few of those markers likely survived your HVG filter — check [g for g in basal+clas if g in X.columns] first. If coverage is thin, score on the un-HVG-filtered logx instead, since marker scoring doesn't need the variance selection.

In [89]:
labels = cc[k]["labels"]
sig    = mc.cluster_signatures(X, labels)
sig[1]

{'stat': A1CF       3.976
 AACS       2.795
 AADAC     -3.667
 AATK       4.560
 ABAT       1.176
            ...  
 ZNRF1     -1.111
 ZNRF2     -0.720
 ZSCAN29   -1.132
 ZSWIM5     2.543
 ZWINT     -5.574
 Length: 2000, dtype: float32,
 'lfc': A1CF       2.104
 AACS       1.056
 AADAC     -2.594
 AATK       1.976
 ABAT       1.392
            ...  
 ZNRF1     -0.497
 ZNRF2     -0.258
 ZSCAN29   -0.405
 ZSWIM5     1.149
 ZWINT     -1.926
 Length: 2000, dtype: float32,
 'up': ['PPFIA3',
  'SCAMP5',
  'KCNH2',
  'FOXP4',
  'DYRK1B',
  'SEZ6L2',
  'SRCIN1',
  'SSTR1',
  'CELSR3',
  'GIPR',
  'FOXA2',
  'RGL3',
  'DUSP8',
  'SMIM6',
  'LRRC61',
  'SYNE4',
  'GOLGA7B',
  'FBXO41',
  'CAMSAP3',
  'AATK',
  'EEF1A2',
  'FAM174B',
  'SLC25A29',
  'UBE2M',
  'TPPP',
  'GABRB3',
  'RAB17',
  'MED25',
  'CMTM8',
  'ASPHD1',
  'A1CF',
  'HSPBP1',
  'REPIN1',
  'MANEAL',
  'GPRIN1',
  'SUSD4',
  'RAP1GAP2',
  'FIZ1',
  'SDHAF1',
  'DHRS4-AS1',
  'HNF1A',
  'CACFD1',
  'DDC',
  'MYEF2',
  'MED29',
 

In [90]:
"; ".join(sig[1]['stat'].index.to_list())

'A1CF; AACS; AADAC; AATK; ABAT; ABCA12; ABCA7; ABCB9; ABCC3; ABCC6; ABCD3; ABHD11; ABHD17C; ABHD2; ABHD3; ABLIM1; ABLIM2; ABLIM3; ABTB2; AC019117.1; ACACA; ACBD5; ACER2; ACHE; ACOT11; ACOT7; ACOX1; ACP6; ACSF2; ACSL5; ACSM3; ACSS1; ACSS2; ACTN4; ACY3; ACYP1; ADAM10; ADAM15; ADAM28; ADAM8; ADAM9; ADAP1; ADCK5; ADH6; ADM2; ADORA2B; ADPRHL1; ADTRP; AFAP1-AS1; AFAP1L2; AFG3L2; AFMID; AFTPH; AGAP9; AGFG1; AGFG2; AGMO; AGPAT2; AGPAT5; AGR2; AGR3; AGRN; AHCYL2; AHNAK2; AHR; AK4; AKAP1; AKR1B10; AKR1C1; AKR1C2; AKR1C3; AKR7A3; ALAS1; ALCAM; ALDH1L1; ALDH3A1; ALDH3B1; ALDOB; ALOX5; ALS2CL; AMIGO2; AMMECR1; AMN; AMOT; ANG; ANK1; ANK3; ANKEF1; ANKRD22; ANKRD36C; ANKRD9; ANKS4B; ANLN; ANO5; ANO8; ANO9; ANPEP; ANXA10; ANXA13; ANXA2; ANXA3; ANXA4; ANXA8; AOC1; AP001372.2; AP1AR; AP1G2; AP1M2; AP1S1; AP1S3; APCS; APLP2; APOL1; AQP3; AQP5; ARAP2; AREG; ARHGAP11A; ARHGAP12; ARHGAP26; ARHGAP27; ARHGAP32; ARHGAP5; ARHGEF10L; ARHGEF16; ARHGEF28; ARHGEF35; ARHGEF37; ARHGEF38; ARHGEF39; ARHGEF5; ARID3B; ARL

In [91]:
"; ".join(sig[2]['stat'].index.to_list())

'A1CF; AACS; AADAC; AATK; ABAT; ABCA12; ABCA7; ABCB9; ABCC3; ABCC6; ABCD3; ABHD11; ABHD17C; ABHD2; ABHD3; ABLIM1; ABLIM2; ABLIM3; ABTB2; AC019117.1; ACACA; ACBD5; ACER2; ACHE; ACOT11; ACOT7; ACOX1; ACP6; ACSF2; ACSL5; ACSM3; ACSS1; ACSS2; ACTN4; ACY3; ACYP1; ADAM10; ADAM15; ADAM28; ADAM8; ADAM9; ADAP1; ADCK5; ADH6; ADM2; ADORA2B; ADPRHL1; ADTRP; AFAP1-AS1; AFAP1L2; AFG3L2; AFMID; AFTPH; AGAP9; AGFG1; AGFG2; AGMO; AGPAT2; AGPAT5; AGR2; AGR3; AGRN; AHCYL2; AHNAK2; AHR; AK4; AKAP1; AKR1B10; AKR1C1; AKR1C2; AKR1C3; AKR7A3; ALAS1; ALCAM; ALDH1L1; ALDH3A1; ALDH3B1; ALDOB; ALOX5; ALS2CL; AMIGO2; AMMECR1; AMN; AMOT; ANG; ANK1; ANK3; ANKEF1; ANKRD22; ANKRD36C; ANKRD9; ANKS4B; ANLN; ANO5; ANO8; ANO9; ANPEP; ANXA10; ANXA13; ANXA2; ANXA3; ANXA4; ANXA8; AOC1; AP001372.2; AP1AR; AP1G2; AP1M2; AP1S1; AP1S3; APCS; APLP2; APOL1; AQP3; AQP5; ARAP2; AREG; ARHGAP11A; ARHGAP12; ARHGAP26; ARHGAP27; ARHGAP32; ARHGAP5; ARHGEF10L; ARHGEF16; ARHGEF28; ARHGEF35; ARHGEF37; ARHGEF38; ARHGEF39; ARHGEF5; ARID3B; ARL

p = 4.8e-48 with n=6 is not credible. Welch t with ~5 df cannot produce that unless the within-group variance of the 6 is near zero. That's the fingerprint of prior domination: BayesPrism shrank all six toward the same reference profile, so they're nearly identical to each other. Tiny SE → exploding t → absurd p. Check directly:

In [92]:
small = labels[labels==1].index
big   = labels[labels==2].index
pd.DataFrame({
    "sd_small": X.loc[small].std(axis=0),
    "sd_big":   X.loc[big].std(axis=0),
}).describe().round(3)

,sd_small,sd_big
count,2000.000,2000.000
mean,1.019,0.936
std,0.649,0.430
min,0.000,0.312
25%,0.557,0.641
50%,0.837,0.824
75%,1.316,1.103
max,5.261,3.658


### Welch test

In [93]:
labs1 = labels[labels==1].index
labs2 = labels[labels==2].index

X1 = X.loc[labs1]
X2 = X.loc[labs2]

X1.shape, X2.shape

((6, 2000), (111, 2000))

In [94]:
X1.iloc[:5, :10]

,A1CF,AACS,AADAC,AATK,ABAT,ABCA12,ABCA7,ABCB9,ABCC3,ABCC6
T-TCGA-3A-A9IL,5.575,7.075,1.807,8.059,5.876,0.494,7.629,5.042,6.422,1.829
T-TCGA-3A-A9IN,5.359,6.804,3.016,7.399,6.519,0.212,6.941,4.283,4.060,3.411
T-TCGA-3A-A9IS,6.927,7.778,0.000,6.636,9.488,0.152,6.178,4.314,2.796,4.440
T-TCGA-2L-AAQM,5.741,7.466,1.078,7.554,2.208,0.559,6.867,4.470,3.009,0.816
T-TCGA-3A-A9IR,5.133,6.687,0.000,6.818,7.317,0.000,7.178,4.236,1.610,1.980


In [95]:
X1.mean(axis=0)

A1CF       5.945
AACS       7.258
AADAC      0.984
AATK       7.192
ABAT       6.522
           ...  
ZNRF1      4.824
ZNRF2      4.726
ZSCAN29    4.989
ZSWIM5     4.788
ZWINT      3.212
Length: 2000, dtype: float32

In [96]:
m1, m2 = X1.mean(axis=0), X2.mean(axis=0)      # Series, one value per gene
v1, v2 = X1.var(axis=0, ddof=1), X2.var(axis=0, ddof=1)
n1, n2 = len(X1), len(X2)

In [97]:
from scipy import stats

se    = np.sqrt(v1/n1 + v2/n2)
tw    = (m1 - m2) / se
dfree = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
# Survival function (also defined as 1 - cdf, but sf is sometimes more accurate).
p = 2 * stats.t.sf(np.abs(tw), dfree)
p

array([5.98986502e-04, 2.34204627e-03, 2.94197589e-03, ...,
       5.90306330e-02, 5.71928075e-03, 1.51290032e-05])

In [98]:
pd.DataFrame({
    "p": p,
}).describe()

,p
count,2.000e+03
mean,1.206e-01
std,2.254e-01
min,4.798e-48
25%,6.345e-04
50%,8.469e-03
75%,1.127e-01
max,9.984e-01


In [99]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')

fdr_cutoff = 0.05
lfc_cutoff = 1

dft_all = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.lfc.abs() >= lfc_cutoff)]
print(dft_all.shape)

dft_all.head(2)

(2186, 12)


,cluster,gene,lfc,stat,p_nominal,fdr_nominal,sd_cluster,sd_rest,welch_df,n_cluster,direction,rank
0,1,MUC17,-6.116,-13.057,1.520e-32,3.039e-29,0.035,3.015,82.436,6,down,1
1,1,PGC,-5.744,-10.838,2.765e-28,2.765e-25,0.063,3.658,99.677,6,down,2


In [ ]:
p = dft_all.loc[dft_all.cluster == 1, "p_nominal"]
p.describe()

In [ ]:
p.hist()

### Review clusters - detailed

In [ ]:
small_out = labels[labels == 1].index
X2, d2 = mc.prepare_malignant_matrix(
    keep_genes=mc.program1_panel, drop_pattern=r"^N-",
    keep_samples=[s for s in mc.ms.Z.index if s not in set(small_out)])
cc2 = mc.consensus_cluster(X2)
k2  = mc.choose_k(cc2)
print(k2) 
cc2[k2]["labels"].value_counts().to_dict()

In [ ]:
mc.cluster_summary(cc2)

If min_frac is tiny at every k, don't pick a k. Find the QC axis instead:

In [ ]:
d2["sample_total_Z"].sort_values().head(15)
mc.ms.theta_mal[X2.index].sort_values().head(15)
ties = X2.round(6).apply(lambda c: c.duplicated(keep=False)).mean(axis=1)
ties.sort_values(ascending=False).head(15)

### Critic

In your notebook, cell 72's output showed 'n': 6 and 'n': 119 - the n field cluster_signatures records for each cluster. 

So choose_k picked k=2, but the two groups were 6 samples and 119 samples, not two comparable halves.

In [ ]:
sig[1]['n'], sig[2]['n']

### Each signature

In [ ]:

clu=2
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
clu=1
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
sig[clu]['up']

### signature_table()

In [ ]:
len(labels), labels

In [ ]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')
dft['abs_lfc'] = dft['lfc'].abs()

fdr_cutoff = 0.05
lfc_cutoff = 1

dft = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.abs_lfc >= lfc_cutoff)]
dft = dft.sort_values('fdr_nominal', ascending=True)

dft.shape

In [ ]:
dft.fdr_nominal.hist()

In [ ]:
dft.head(6)

In [ ]:
genes_sel = sig[clu]['up']

df1 = dft[dft.gene.isin(genes_sel)]
print(f"Number of upregulated genes in cluster {clu}: {df1.shape[0]}")
df1

In [ ]:
genes_clu = np.unique(df1.gene)
print(len(genes_clu))


In [ ]:
genes_clu_in = [x for x in genes_clu if x in df_pivot.index]
df2 = df_pivot.loc[genes_clu_in]

print(df2.shape)
df2.T

### WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score)

WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score) are core metrics used in the LINCS and Connectivity Map (CMap) pipelines to compare query gene signatures against reference expression profiles. 

- WTCS measures signature similarity from −1 to 1
- NCS normalizes these scores within specific cell lines and perturbagen types.

In [ ]:
cond

In [ ]:
cond.index.is_unique 

In [ ]:
cond = cond[~cond.index.duplicated()].loc[df_pivot.columns]
print(cond.index.is_unique)
cond.head(3)

In [ ]:
R1 = mc.score_clusters_vs_tahoe(sig, df_pivot, cond)
R1

In [ ]:
len(R1.targets.unique()), R1.targets.unique()[:20]

In [ ]:
target_list = R1.targets.unique()
len(target_list), len(genes_clu)

In [ ]:
[x for x in target_list if x in genes_clu]

In [ ]:
R1a = R1[ (R1.cluster == clu) & (R1.targets.isin(genes_clu)) & (R1.wtcs.abs() > 0.1) ]
print(R1a.shape)
R1a

In [ ]:
R1a.targets.unique()

In [ ]:
R1a.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R1a.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
out = mc.run(ks=range(2, 9), drop_pattern=r"^N-", min_share=0.3)     # tune from diagnose_filters()
R   = mc.score_clusters_vs_tahoe(out["signatures"], df_pivot, cond)

In [ ]:
R.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
# mc.root_mprog_tahoe = create_dir(mc.root_mprog_cluster / "tahoe")
mc.root_mprog_tahoe

In [ ]:
d = mc.root_mprog_tahoe / "metadata" / "pseudobulk_differential_expression"
files = sorted(d.glob("*.parquet"))
len(files), sum(f.stat().st_size for f in files) / 1e9

In [ ]:
cl = pd.read_parquet(mc.root_mprog_tahoe / "metadata"/ "cell_line_metadata.parquet")
cl[cl.Organ=="Pancreas"][["Cell_ID_Cellosaur","cell_name"]]

In [ ]:
d = mc.diagnose_filters()

d["Z_looks_like_counts"], d["Z_median_of_medians"]


In [ ]:
d["by_min_counts"]

In [ ]:
d["by_min_share"]

In [ ]:
d["joint_grid"]